#### Steps to follow:
#### 1. Understand Ground Truth lag, from Part-1 (Waiting For Chargebacks), we concluded that we need to wait 32 days after the payment date in order to understand if its a chargeback or not

#### 2. Determine test set period: The test set should cover a month's worth of data, the end date of the test set should be the latest payment date minus the ground truth lag (32 days) to ensure labels are available.


#### 3. Calculate training set period: The training set should include all the data upto 32 days before start of the test set to ensure that all lables are known by the time model is deployed

In [1]:
#Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

%config InlineBackend.figure_format = 'retina'

In [2]:
df = pd.read_csv("data/payments.csv",index_col=0)
df.head(5)

,merchant_id,buyer_id,transaction_timestamp,payment_amount,chargeback_timestamp
id,,,,,
547601f8-6288-403d-8711-0d4e752f231c,2512b0ea-644d-4440-801d-63ca077a3ea0,6bbc28bd-49b8-424d-89c5-4239d5147e4e,2022-05-22 09:55:54,6352,NaN
4e069be8-e01b-4aee-bf1b-f3ddfc8cde6d,696b3ae8-a375-459b-ac76-cb17e15aec39,d4618c41-c414-4b4f-b439-e4f70b391496,2022-08-08 20:41:07,2810,NaN
bd64067f-5db1-4614-88ee-e678bad2ccce,5032c286-4189-4df1-8d8f-850e3ff44411,b1de3a9b-3688-4fc7-8c1d-87f669d9243e,2022-07-10 18:20:25,5226,NaN
2bebdfe8-f421-4ac5-9706-76938f0c0f64,0ae96c8a-1a2a-4070-8439-1d8e1a741238,e45a8c36-5720-4125-88b8-10e4c745fad4,2022-03-02 18:05:47,725,2022-03-10 03:24:40.790145
393f61b7-e789-41f7-9727-2e7d629e87d0,737032c5-6935-47bb-af7a-2d1e455a77cf,0cba1762-b083-4747-bf28-416906b29d2b,2022-02-12 09:08:37,2437,NaN


In [3]:
#create separate column for transaction_date and transaction_time
df["transaction_date"] = pd.to_datetime(df["transaction_timestamp"]).dt.date
df["transaction_time"] = pd.to_datetime(df["transaction_timestamp"]).dt.time

#create separate column for chargeback_date and chargeback_time
df["chargeback_date"] = pd.to_datetime(df["chargeback_timestamp"]).dt.date
df["chargeback_time"] = pd.to_datetime(df["chargeback_timestamp"]).dt.time

# Calculate time difference in days when chargeback occured
df['days_diff'] = df['chargeback_date'] - df['transaction_date']

In [4]:
# Filter out rows where chargeback occurred
chargebacks = df[df['days_diff'].notnull()]
chargebacks.head(5)

,merchant_id,buyer_id,transaction_timestamp,payment_amount,chargeback_timestamp,transaction_date,transaction_time,chargeback_date,chargeback_time,days_diff
id,,,,,,,,,,
2bebdfe8-f421-4ac5-9706-76938f0c0f64,0ae96c8a-1a2a-4070-8439-1d8e1a741238,e45a8c36-5720-4125-88b8-10e4c745fad4,2022-03-02 18:05:47,725,2022-03-10 03:24:40.790145,2022-03-02,18:05:47,2022-03-10,03:24:40.790145,8 days
1486d307-67c6-4733-9d5a-b535b8f70b84,eaec9229-e57a-40c9-b145-5d7a712a5f83,451267ce-abc1-4716-81de-be612fce4c1e,2022-08-22 03:06:53,10892,2022-09-04 02:37:47.039960,2022-08-22,03:06:53,2022-09-04,02:37:47.039960,13 days
59e646d6-3efe-4a33-a942-42b600a49ebd,3a1aef79-d65b-4395-b39d-1a73d4494c82,422dec8e-e4fc-4eb6-931c-1b886672d1a4,2022-04-07 08:39:46,2737,2022-04-27 13:00:19.904650,2022-04-07,08:39:46,2022-04-27,13:00:19.904650,20 days
7f3d28a9-27c1-4485-b534-82cdc08fa21e,1908e756-b4a4-4bd9-a30c-1988b771b61a,a33159fa-2fd8-4cc2-b99e-a74cc4589bc2,2022-08-18 02:44:17,1004,2022-08-21 22:34:24.538558,2022-08-18,02:44:17,2022-08-21,22:34:24.538558,3 days
3cd9c33e-ce5a-4547-804d-ec2fa7fa1b34,e381d43b-8435-465e-a297-57618e0053a7,9dd571a9-f485-414e-a048-883185980365,2022-08-26 05:18:52,402,2022-09-06 19:18:22.118781,2022-08-26,05:18:52,2022-09-06,19:18:22.118781,11 days


In [20]:
# Constants from part 1: ground truth available after 32 days
L_DAYS = 32
TEST_WINDOW_DAYS = 30  # a month's worth of data

In [34]:
# Find the latest payment transaction date in the dataset
most_recent_trx_date = df["transaction_date"].max()
most_recent_trx_date

datetime.date(2022, 8, 31)

In [35]:
# Find the latest payment transaction date in the dataset
oldest_trx_date = df["transaction_date"].min()
oldest_trx_date

datetime.date(2022, 1, 1)

In [22]:
## Calculate the end of the test set as the latest timestamp minus ground truth lag
test_end = most_recent_trx_date - pd.Timedelta(days=L_DAYS) 
test_end

datetime.date(2022, 7, 30)

In [23]:
# Calculate the start of the test set as 30 days before test_end
test_start = test_end - pd.Timedelta(days=TEST_WINDOW_DAYS)
test_start

datetime.date(2022, 6, 30)

In [25]:
#Training data must end L_DAYS before the test set starts to ensure ground truth is known
training_end = test_start - pd.Timedelta(days = L_DAYS)
training_end

datetime.date(2022, 5, 29)

In [27]:
# Split the data into training and test sets
df_train = df[df["transaction_date"] <= training_end]
df_test = df[(df["transaction_date"] >= test_start) & (df["transaction_date"] <= test_end)]

In [29]:
print(f" Training set start date: {df_train['transaction_date'].min()}")

 Training set start date: 2022-01-01


In [30]:
print(f" Training set end date: {df_train['transaction_date'].max()}")

 Training set end date: 2022-05-29


In [32]:
print(f" Test set start date: {df_test['transaction_date'].min()}")

 Test set start date: 2022-06-30


In [33]:
print(f" Test set end date: {df_test['transaction_date'].max()}")

 Test set end date: 2022-07-30
